# 02 - Forgetting ablation (naive / replay / KD / full)

Server-side training only: no detection, no LLM, no mailbox. The 5 reserve MASSIVE intents are added in a fixed order with their real data, from the same V0 and the same seed, so only the loss and the training set change between conditions.\n\nResults are written per version to `outputs/results/<run_id>/v{n}.json`, so a Colab disconnection costs one version, not the whole run.

In [1]:
import os
from google.colab import drive, userdata
drive.mount('/content/drive')
os.environ['WANDB_API_KEY'] = userdata.get('wandbKey')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('anthropic_key')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [3]:
%pip install -q "transformers>=4.48,<5" "huggingface_hub>=0.34,<1" "datasets<3.0.0" anthropic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 112.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.3/527.3 kB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 60.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 84.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 15.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompati

In [ ]:
%ls

drive/  sample_data/


In [3]:
%cd /content/drive/MyDrive/incremental_learning_on_the_edge/src

/content/drive/MyDrive/incremental_learning_on_the_edge/src


In [4]:
!git pull

remote: Enumerating objects: 17, done.
remote: Counting objects: 100% (17/17), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 10 (delta 6), reused 10 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (10/10), 2.82 KiB | 0 bytes/s, done.
From https://github.com/BrunoSantome/incremental_learning_on_the_edge
   f510a48..10e37ca  main       -> origin/main
Updating f510a48..10e37ca
Fast-forward
 src/experiments/ablation.py                   | 114 +++++++++++++++++++++-----
 src/experiments/config.py                     |  10 ++-
 src/experiments/synthetic_cache.py            |  42 ++++++++++
 src/outputs/datasets/horizon_escalations.json |  17 ++++
 4 files changed, 159 insertions(+), 24 deletions(-)
 create mode 100644 src/outputs/datasets/horizon_escalations.json


In [ ]:
from core.configuration import load_config
from core.dataloader import DataClass
from experiments.ablation import reserve_intent_names, CONDITIONS

config = load_config()
print("conditions:", CONDITIONS)
print("intents, in order:", reserve_intent_names(DataClass(), config))
# conditions: {'naive': (False, False), 'replay': (False, True), 'kd': (True, False), 'full': (True, True)}
# intents, in order: ['takeaway_order', 'general_joke', 'recommendation_locations', 'transport_traffic', 'play_podcasts']

`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qanastek/MASSIVE' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.
ERROR:datasets.load:`trust_remote_code` is not supported anymore.
Please check that the Hugging Face dataset 'qanastek/MASSIVE' isn't based on a loading script and remove `trust_remote_code`.
If the dataset is based on a loading script, please ask the dataset author to remove it and convert it to a standard format like Parquet.


conditions: {'naive': (False, False, False), 'replay': (False, True, True), 'kd': (True, False, False), 'full': (True, True, True)}


README.md:   0%|          | 0.00/34.1k [00:00<?, ?B/s]

MASSIVE.py:   0%|          | 0.00/32.3k [00:00<?, ?B/s]

RuntimeError: Dataset scripts are no longer supported, but found MASSIVE.py

In [ ]:
# !python -m experiments.ablation  --conditions naive replay kd full --seeds 42

### Ablation runs seed 42

In [ ]:
!python -m experiments.ablation --report --conditions naive --seeds 42

2026-10-01 16:30:30.496938: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Filter: 100% 1064/1064 [00:00<00:00, 80405.37 examples/s]
Map: 100% 914/914 [00:00<00:00, 9525.33 examples/s]
evaluating test set: 100% 29/29 [00:07<00:00,  3.66it/s]
V0: has 15 intents on the test split
Filter: 100% 1064/1064 [00:00<00:00, 71758.61 examples/s]
Map: 100% 936/936 [00:00<00:00, 9232.99 examples/s]
evaluating test set: 100% 30/30 [00:01<00:00, 25.30it/s]
V1: has 16 intents on the test split
Filter: 100% 1064/1064 [00:00<00:00, 115543.17 examples/s]
Map: 100% 955/955 [00:00<00:00, 10874.21 examples/s]
evaluating test set: 100% 30/30 [00:01<00:00, 25.07it/s]
V2: has 17 intents on the test split
Filter: 100% 1064/1064 [00:00<00:00, 121039.86 examples/s]
Map: 10

In [ ]:
!python -m experiments.ablation --report --conditions replay  --seeds 42

2026-10-01 16:44:24.913519: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
evaluating test set: 100% 29/29 [00:06<00:00,  4.82it/s]
V0: has 15 intents on the test split
Map: 100% 936/936 [00:00<00:00, 9241.43 examples/s]
evaluating test set: 100% 30/30 [00:01<00:00, 24.02it/s]
V1: has 16 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 24.38it/s]
V2: has 17 intents on the test split
evaluating test set: 100% 31/31 [00:01<00:00, 23.79it/s]
V3: has 18 intents on the test split
evaluating test set: 100% 32/32 [00:01<00:00, 23.76it/s]
V4: has 19 intents on the test split
evaluating test set: 100% 34/34 [00:01<00:00, 23.92it/s]
V5: has 20 intents on the test split

=== ablation_replay_real_K70_a0.0_w0.3_s42 ===

retention (old 

In [ ]:
!python -m experiments.ablation --report --conditions full --seeds 42

2026-10-01 16:46:12.064539: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
evaluating test set: 100% 29/29 [00:07<00:00,  3.78it/s]
V0: has 15 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 24.23it/s]
V1: has 16 intents on the test split
Map: 100% 955/955 [00:00<00:00, 8008.59 examples/s]
evaluating test set: 100% 30/30 [00:01<00:00, 22.74it/s]
V2: has 17 intents on the test split
evaluating test set: 100% 31/31 [00:01<00:00, 24.10it/s]
V3: has 18 intents on the test split
evaluating test set: 100% 32/32 [00:01<00:00, 22.67it/s]
V4: has 19 intents on the test split
evaluating test set: 100% 34/34 [00:01<00:00, 24.07it/s]
V5: has 20 intents on the test split

=== ablation_full_real_K70_a0.2_w0.3_s42 ===

retention (old in

In [ ]:
!python -m experiments.ablation --conditions kd --seeds 42


=== ablation_kd_real_K70_a0.2_w0.3_s42 ===
intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'transport_traffic', 'play_podcasts']
v1: adding 'takeaway_order' [kd=True, replay=False]
2026-10-01 16:47:38.409882: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bruno-santome-antolin (bruno-santome-antolin-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run h74etdfo (0.2s)
wandb: ⣯ setting up run h7

In [ ]:
!python -m experiments.ablation --report --conditions kd --seeds 42

2026-10-01 17:49:22.080642: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
evaluating test set: 100% 29/29 [00:06<00:00,  4.75it/s]
V0: has 15 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 24.39it/s]
V1: has 16 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 24.67it/s]
V2: has 17 intents on the test split
evaluating test set: 100% 31/31 [00:01<00:00, 25.71it/s]
V3: has 18 intents on the test split
Map: 100% 1001/1001 [00:00<00:00, 8975.02 examples/s]
evaluating test set: 100% 32/32 [00:01<00:00, 25.74it/s]
V4: has 19 intents on the test split
evaluating test set: 100% 34/34 [00:01<00:00, 25.84it/s]
V5: has 20 intents on the test split

=== ablation_kd_real_K70_a0.2_w0.3_s42 ===

retention (old in

### Ablation runs seed 43

In [ ]:
!python -m experiments.ablation --conditions naive kd replay full --seeds 43


=== ablation_naive_real_K70_a0.0_w0.3_s43 ===
intents: ['takeaway_order', 'general_joke', 'recommendation_locations', 'transport_traffic', 'play_podcasts']
v1: adding 'takeaway_order' [kd=False, replay=False]
2026-10-01 17:50:16.349229: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: bruno-santome-antolin (bruno-santome-antolin-city-university-of-london) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: ⢿ Waiting for wandb.init()...
wandb: ⣻ Waiting for wandb.init()...
wandb: ⣽ Waiting for wandb.init()...
wandb: ⣾ Waiting for wandb.init()...
wandb: ⣷ setting up run cf7ojhxe (0.2s)
wandb: ⣯ setting up ru

In [ ]:
!python -m experiments.ablation --report --conditions naive kd replay full --seeds 43

Filter: 100% 11514/11514 [00:00<00:00, 102740.16 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 104229.38 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 102557.25 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 129483.19 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 119210.80 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 106098.37 examples/s]
Map: 100% 3622/3622 [00:00<00:00, 5291.11 examples/s]
Map: 100% 914/914 [00:00<00:00, 8493.28 examples/s]
Map: 100% 657/657 [00:00<00:00, 6394.72 examples/s]
Casting the dataset: 100% 3622/3622 [00:00<00:00, 572863.57 examples/s]
Casting the dataset: 100% 914/914 [00:00<00:00, 196544.16 examples/s]
Casting the dataset: 100% 657/657 [00:00<00:00, 138978.10 examples/s]
Casting the dataset: 100% 690/690 [00:00<00:00, 65735.47 examples/s]
Casting the dataset: 100% 150/150 [00:00<00:00, 24302.60 examples/s]
Casting the dataset: 100% 122/122 [00:00<00:00, 18960.47 examples/s]
Map: 100% 690/690 [00:00<00:00, 12521.29 examples/s]
Map: 100% 15

### Synthetic data evaluation ablation full vs replay only

In [ ]:
!python -m experiments.ablation --data synthetic --conditions replay full --seeds 42

takeaway_order: generated 134 utterances from 'please order some sushi for dinner' -> /content/drive/My Drive/incremental_learning_on_the_edge/src/outputs/datasets/synth_utterances_s42.json
Filter: 100% 690/690 [00:00<00:00, 94201.87 examples/s]
general_joke: generated 138 utterances from 'make me laugh' -> /content/drive/My Drive/incremental_learning_on_the_edge/src/outputs/datasets/synth_utterances_s42.json
Filter: 100% 690/690 [00:00<00:00, 106258.99 examples/s]
recommendation_locations: generated 134 utterances from 'find me a nice restaurant for dinner' -> /content/drive/My Drive/incremental_learning_on_the_edge/src/outputs/datasets/synth_utterances_s42.json
Filter: 100% 690/690 [00:00<00:00, 112758.89 examples/s]
transport_traffic: generated 133 utterances from 'just how bad is traffic on my commute' -> /content/drive/My Drive/incremental_learning_on_the_edge/src/outputs/datasets/synth_utterances_s42.json
Filter: 100% 690/690 [00:00<00:00, 115329.15 examples/s]
play_podcasts: gen

In [ ]:
!python -m experiments.ablation --report --data synthetic --conditions replay full --seeds 42

2026-10-01 18:52:50.659791: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
evaluating test set: 100% 29/29 [00:07<00:00,  4.04it/s]
V0: has 15 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 24.96it/s]
V1: has 16 intents on the test split
evaluating test set: 100% 30/30 [00:01<00:00, 25.05it/s]
V2: has 17 intents on the test split
evaluating test set: 100% 31/31 [00:01<00:00, 24.59it/s]
V3: has 18 intents on the test split
evaluating test set: 100% 32/32 [00:01<00:00, 23.70it/s]
V4: has 19 intents on the test split
evaluating test set: 100% 34/34 [00:01<00:00, 24.39it/s]
V5: has 20 intents on the test split

=== ablation_replay_synthetic_K70_a0.0_w0.3_s42 ===

retention (old intents)
                         V0        V1 

synthetic seed 43 runs

In [ ]:
!python -m experiments.ablation --data synthetic --conditions replay full --seeds 43

Generating train split: 11514 examples [00:01, 6214.84 examples/s]
Generating validation split: 2033 examples [00:00, 3029.53 examples/s]
Generating test split: 2974 examples [00:01, 2679.93 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 123133.84 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 111768.93 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 102333.25 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 122617.99 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 106816.87 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 104065.46 examples/s]
Map: 100% 3622/3622 [00:00<00:00, 7798.33 examples/s]
Map: 100% 914/914 [00:00<00:00, 8157.09 examples/s]
Map: 100% 657/657 [00:00<00:00, 8359.70 examples/s]
Casting the dataset: 100% 3622/3622 [00:00<00:00, 592548.92 examples/s]
Casting the dataset: 100% 914/914 [00:00<00:00, 142169.25 examples/s]
Casting the dataset: 100% 657/657 [00:00<00:00, 146080.24 examples/s]
Casting the dataset: 100% 690/690 [00:00<00:00, 58470.78 examples/s]


In [5]:
!python -m experiments.ablation --report --data synthetic --conditions replay full --seeds 43

Generating train split: 11514 examples [00:01, 5840.13 examples/s]
Generating validation split: 2033 examples [00:00, 2536.71 examples/s]
Generating test split: 2974 examples [00:00, 3370.91 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 209627.81 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 193435.17 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 170444.95 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 221190.64 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 192560.24 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 179697.80 examples/s]
Map: 100% 3622/3622 [00:00<00:00, 8392.72 examples/s]
Map: 100% 914/914 [00:00<00:00, 7537.69 examples/s]
Map: 100% 657/657 [00:00<00:00, 8116.41 examples/s]
Casting the dataset: 100% 3622/3622 [00:00<00:00, 573642.30 examples/s]
Casting the dataset: 100% 914/914 [00:00<00:00, 189566.03 examples/s]
Casting the dataset: 100% 657/657 [00:00<00:00, 147487.57 examples/s]
Casting the dataset: 100% 690/690 [00:00<00:00, 51730.62 examples/s]


### Ablation Long  Horizon tasks 15 intents increment Full (KD + replay) vs Replay-only

In [5]:
!python -m experiments.ablation --escalations outputs/datasets/horizon_escalations.json --conditions replay full --seeds 42

Generating train split: 11514 examples [00:01, 6405.29 examples/s]
Generating validation split: 2033 examples [00:00, 2786.85 examples/s]
Generating test split: 2974 examples [00:00, 3641.98 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 225698.77 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 199122.98 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 160222.10 examples/s]
Filter: 100% 11514/11514 [00:00<00:00, 229491.23 examples/s]
Filter: 100% 2974/2974 [00:00<00:00, 194946.71 examples/s]
Filter: 100% 2033/2033 [00:00<00:00, 186256.74 examples/s]
Map: 100% 3622/3622 [00:00<00:00, 13291.61 examples/s]
Map: 100% 914/914 [00:00<00:00, 13528.39 examples/s]
Map: 100% 657/657 [00:00<00:00, 11950.78 examples/s]
Casting the dataset: 100% 3622/3622 [00:00<00:00, 880682.27 examples/s]
Casting the dataset: 100% 914/914 [00:00<00:00, 288522.15 examples/s]
Casting the dataset: 100% 657/657 [00:00<00:00, 203354.57 examples/s]
Casting the dataset: 100% 690/690 [00:00<00:00, 73916.94 examples/

In [7]:
!python -m experiments.ablation --report --escalations outputs/datasets/horizon_escalations.json --conditions replay full --seeds 42

Casting the dataset: 100% 93/93 [00:00<00:00, 43936.73 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 11317.60 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 11592.88 examples/s]
Map: 100% 93/93 [00:00<00:00, 17816.31 examples/s]
Casting the dataset: 100% 93/93 [00:00<00:00, 49951.37 examples/s]
Map: 100% 20/20 [00:00<00:00, 7145.32 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 11480.24 examples/s]
Map: 100% 20/20 [00:00<00:00, 6690.01 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 11101.92 examples/s]
Casting the dataset: 100% 95/95 [00:00<00:00, 52490.96 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 11252.32 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 8565.92 examples/s]
Map: 100% 95/95 [00:00<00:00, 19228.78 examples/s]
Casting the dataset: 100% 95/95 [00:00<00:00, 52004.55 examples/s]
Map: 100% 20/20 [00:00<00:00, 6699.63 examples/s]
Casting the dataset: 100% 20/20 [00:00<00:00, 10123.83 examples/s]
Map: 100% 20

### Inspect the saved results

In [ ]:
import json, glob
for path in sorted(glob.glob("outputs/results/ablation_*/v*.json")):
    print(path)

outputs/results/ablation_full_real_K70_a0.2_w0.3_s42/v1.json
outputs/results/ablation_full_real_K70_a0.2_w0.3_s42/v2.json
outputs/results/ablation_full_real_K70_a0.2_w0.3_s42/v3.json
outputs/results/ablation_full_real_K70_a0.2_w0.3_s42/v4.json
outputs/results/ablation_full_real_K70_a0.2_w0.3_s42/v5.json
outputs/results/ablation_naive_real_K70_a0.0_w0.3_s42/v1.json
outputs/results/ablation_naive_real_K70_a0.0_w0.3_s42/v2.json
outputs/results/ablation_naive_real_K70_a0.0_w0.3_s42/v3.json
outputs/results/ablation_naive_real_K70_a0.0_w0.3_s42/v4.json
outputs/results/ablation_naive_real_K70_a0.0_w0.3_s42/v5.json
outputs/results/ablation_replay_real_K70_a0.0_w0.3_s42/v1.json
outputs/results/ablation_replay_real_K70_a0.0_w0.3_s42/v2.json
outputs/results/ablation_replay_real_K70_a0.0_w0.3_s42/v3.json
outputs/results/ablation_replay_real_K70_a0.0_w0.3_s42/v4.json
outputs/results/ablation_replay_real_K70_a0.0_w0.3_s42/v5.json
